# 100 · Data structures — procedural layer

**Mnemonic:** 1 looks like a single node or cell — data as items chained one to the next.

**Codes covered:** 104, 116, 127, 132, 143, 156, 167, 173, 182, 195.

**Method:** read each code cell and PREDICT the exact output, then run it and compare. The matrix holds the imagery; this notebook engraves the net effects — what each structure actually does to your data when you press Enter.

## 104 · Amortized O(1) append

Occasional O(n) resize copies are spread across the many cheap appends before them, averaging to constant time per append.

*A mortgage: one huge house price paid off in tiny monthly installments — each append pays a small installment toward the rare big copy.*

**Watch:** the byte size only jumps at a handful of lengths, and the gaps between jumps widen geometrically. Every append between two jumps costs nothing extra — those cheap appends prepay the rare copy.

In [1]:
import sys

lst = []
prev = sys.getsizeof(lst)
print(f"len=0   size={prev} bytes")
for i in range(1, 65):
    lst.append(i)
    size = sys.getsizeof(lst)
    if size != prev:
        print(f"len={len(lst):<3} size jumped {prev} -> {size} bytes")
        prev = size
print("64 appends, only a few resizes: the copies average out to O(1) per append.")

len=0   size=56 bytes
len=1   size jumped 56 -> 88 bytes
len=5   size jumped 88 -> 120 bytes
len=9   size jumped 120 -> 184 bytes
len=17  size jumped 184 -> 248 bytes
len=25  size jumped 248 -> 312 bytes
len=33  size jumped 312 -> 376 bytes
len=41  size jumped 376 -> 472 bytes
len=53  size jumped 472 -> 568 bytes
64 appends, only a few resizes: the copies average out to O(1) per append.


## 116 · Floyd cycle detection

If a fast pointer (2 steps) and a slow pointer (1 step) ever land on the same node, the list has a cycle; on a straight list fast hits null first.

*Tortoise and hare on a circular racetrack: on a loop the hare must eventually lap and collide with the tortoise; on a straight road the hare just vanishes over the horizon.*

**Watch:** the same function flips True/False as we bend the last node's pointer back and then straighten it again — O(1) extra space, no visited set.

In [2]:
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None

def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

nodes = [Node(v) for v in range(1, 6)]
for a, b in zip(nodes, nodes[1:]):
    a.next = b

print("straight list 1->2->3->4->5:", has_cycle(nodes[0]))
nodes[-1].next = nodes[1]  # bend 5's pointer back to 2
print("cycle added   5 -> 2      :", has_cycle(nodes[0]))
nodes[-1].next = None      # straighten it again
print("cycle broken              :", has_cycle(nodes[0]))

straight list 1->2->3->4->5: False
cycle added   5 -> 2      : True
cycle broken              : False


## 127 · Queue from two stacks

Push into an inbox stack; when the outbox stack is empty, pour the whole inbox into it, reversing order so pops come out FIFO.

*Two Pringles cans: drop chips into can A; to serve, flip A upside down over can B — the oldest chip lands on top of B, ready to serve first.*

**Watch:** three LIFO pushes come back out in FIFO order 1, 2, 3 — two reversals cancel. Each element moves at most twice, so it is amortized O(1) per operation.

In [3]:
class Queue2Stacks:
    def __init__(self):
        self.inbox, self.outbox = [], []
    def enqueue(self, x):
        self.inbox.append(x)
    def dequeue(self):
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())  # the flip: LIFO reversed
        return self.outbox.pop()

q = Queue2Stacks()
for x in (1, 2, 3):
    q.enqueue(x)
    print("enqueue", x, "-> inbox:", q.inbox, "outbox:", q.outbox)
for _ in range(3):
    print("dequeue ->", q.dequeue(), "| inbox:", q.inbox, "outbox:", q.outbox)
print("In 1,2,3 -> out 1,2,3: FIFO built from two LIFOs.")

enqueue 1 -> inbox: [1] outbox: []
enqueue 2 -> inbox: [1, 2] outbox: []
enqueue 3 -> inbox: [1, 2, 3] outbox: []
dequeue -> 1 | inbox: [] outbox: [3, 2]
dequeue -> 2 | inbox: [] outbox: [3]
dequeue -> 3 | inbox: [] outbox: []
In 1,2,3 -> out 1,2,3: FIFO built from two LIFOs.


## 132 · Hash collision

Two different keys landing in the same bucket — inevitable by the pigeonhole principle, so every table needs a resolution strategy.

*Ten pigeons, nine pigeonholes: two pigeons must share one hole, feathers flying. Two strangers' coats both hashed to cubby 47.*

**Watch:** every instance hashes to 7, so all of them land in the same bucket — yet the set stays perfectly correct, because `__eq__` tells colliding keys apart inside the bucket. Collisions cost speed (toward O(n)), never correctness.

In [4]:
class Badge:
    def __init__(self, name):
        self.name = name
    def __hash__(self):
        return 7  # every badge collides on purpose
    def __eq__(self, other):
        return isinstance(other, Badge) and self.name == other.name

a, b = Badge("Ada"), Badge("Bob")
print("hash(a) =", hash(a), "| hash(b) =", hash(b), "-> both land in the same bucket")
s = {a, b}
print("set size          :", len(s), "(collision did not merge them)")
print("members           :", sorted(x.name for x in s))
print("Badge('Ada') in s :", Badge("Ada") in s)
print("Badge('Eve') in s :", Badge("Eve") in s)
print("Same bucket, disambiguated by ==: correctness survives, only speed degrades.")

hash(a) = 7 | hash(b) = 7 -> both land in the same bucket
set size          : 2 (collision did not merge them)
members           : ['Ada', 'Bob']
Badge('Ada') in s : True
Badge('Eve') in s : False
Same bucket, disambiguated by ==: correctness survives, only speed degrades.


## 143 · In-order traversal

Visit the left subtree, then the node, then the right subtree; on a BST this yields every key in ascending sorted order.

*Touring the branching library: read the entire left wing, then the librarian's own book, then the right wing — you exit having read all titles in perfect alphabetical order.*

**Watch:** values inserted in shuffled order come back out sorted. The tree's shape encodes the order; left-node-right just reads it off.

In [5]:
import random

def insert(tree, v):
    if tree is None:
        return {"val": v, "left": None, "right": None}
    side = "left" if v < tree["val"] else "right"
    tree[side] = insert(tree[side], v)
    return tree

def inorder(tree, out):
    if tree:
        inorder(tree["left"], out)   # left wing
        out.append(tree["val"])      # the librarian's own book
        inorder(tree["right"], out)  # right wing
    return out

random.seed(42)
values = list(range(1, 16))
random.shuffle(values)
print("insertion order:", values)

root = None
for v in values:
    root = insert(root, v)

visited = inorder(root, [])
print("in-order visit :", visited)
print("sorted?        :", visited == sorted(values))

insertion order: [9, 14, 8, 7, 15, 13, 6, 3, 10, 4, 5, 12, 1, 2, 11]
in-order visit : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
sorted?        : True


## 156 · Trie (prefix tree)

A tree keyed by characters: each root-to-node path spells a prefix, so lookup and autocomplete cost O(word length), not O(word count).

*A choose-your-own-adventure spelling: from the lobby take hallway 'c', then 'a', then 't' — the room lights up 'word!' and corridors continue toward 'cats' and 'catalog'. Say 'trie' as in reTRIEval.*

**Watch:** a prefix query returns the whole word family for the cost of walking the prefix's letters — 'ca' finds four words without ever scanning the word list.

In [6]:
words = ["cat", "cats", "catalog", "car", "dog"]

root = {}
for w in words:
    node = root
    for ch in w:
        node = node.setdefault(ch, {})
    node["$"] = True  # end-of-word marker

def with_prefix(prefix):
    node = root
    for ch in prefix:          # O(len(prefix)) walk, not O(word count)
        if ch not in node:
            return []
        node = node[ch]
    found = []
    def walk(n, path):
        if "$" in n:
            found.append(prefix + path)
        for c in sorted(k for k in n if k != "$"):
            walk(n[c], path + c)
    walk(node, "")
    return found

print("words in trie:", words)
for p in ("ca", "cat", "do", "z"):
    print(f"prefix {p!r:6} ->", with_prefix(p))

words in trie: ['cat', 'cats', 'catalog', 'car', 'dog']
prefix 'ca'   -> ['car', 'cat', 'catalog', 'cats']
prefix 'cat'  -> ['cat', 'catalog', 'cats']
prefix 'do'   -> ['dog']
prefix 'z'    -> []


## 167 · Top-k via size-k heap

Keep a min-heap of the k best seen: a newcomer beating the heap's root replaces it — O(n log k) time, O(k) space, stream-friendly.

*A ten-champion hall of fame where the doorman is the WEAKEST champion: every challenger fights only the doorman — win, and you take his plaque while the new weakest walks to the door.*

**Watch:** a heap that never holds more than 5 numbers produces exactly the same top-5 as sorting all 1000 — the twist is that top-k LARGEST uses a MIN-heap as gatekeeper.

In [7]:
import heapq
import random

random.seed(42)
stream = [random.randint(0, 1_000_000) for _ in range(1000)]

k = 5
heap = []
for x in stream:
    if len(heap) < k:
        heapq.heappush(heap, x)
    elif x > heap[0]:              # challenger beats the weakest champion
        heapq.heapreplace(heap, x)

top_via_heap = sorted(heap, reverse=True)
top_via_sort = sorted(stream, reverse=True)[:k]
print("heap top-5   :", top_via_heap)
print("sorted()[:5] :", top_via_sort)
print("same answer  :", top_via_heap == top_via_sort)
print(f"but the heap held only {k} numbers while streaming all {len(stream)}.")

heap top-5   : [999816, 999744, 998243, 998160, 997409]
sorted()[:5] : [999816, 999744, 998243, 998160, 997409]
same answer  : True
but the heap held only 5 numbers while streaming all 1000.


## 173 · Adjacency list

For each vertex, store a list of its neighbors: O(V+E) space and fast iteration over a node's edges — the default for sparse graphs.

*Every guest at a party holds a slim personal address book listing only their friends: to see Bob's circle you read Bob's booklet, never the whole town census.*

**Watch:** one dict lookup hands you B's neighbors instantly; the edge list makes you walk every edge in the graph to collect the same answer.

In [8]:
adj = {
    "A": ["B", "C"],
    "B": ["C", "D"],
    "C": ["D"],
    "D": [],
}
edge_list = [(u, v) for u, nbrs in adj.items() for v in nbrs]
print("edge list:", edge_list)

print("neighbors of B, adjacency list:", adj["B"], "  <- one O(1) dict lookup")

checked = 0
scan = []
for (u, v) in edge_list:
    checked += 1
    if u == "B":
        scan.append(v)
print("neighbors of B, edge-list scan:", scan, f"  <- walked all {checked} edges")
print("Same answer; the adjacency list read one booklet, the scan read the census.")

edge list: [('A', 'B'), ('A', 'C'), ('B', 'C'), ('B', 'D'), ('C', 'D')]
neighbors of B, adjacency list: ['C', 'D']   <- one O(1) dict lookup
neighbors of B, edge-list scan: ['C', 'D']   <- walked all 5 edges
Same answer; the adjacency list read one booklet, the scan read the census.


## 182 · Union-find (disjoint sets)

Tracks a partition of items into groups: find(x) walks to the group's root representative; union(a,b) merges two groups by linking roots.

*Clan totems: every villager points at a parent villager, chains ending at the clan chief; a marriage (union) simply makes one chief bow to the other, fusing whole clans in one gesture.*

**Watch:** six unions collapse ten singletons into four components, each merge a single pointer change — and path compression flattens the chains as find() walks them.

In [9]:
parent = list(range(10))  # everyone starts as their own chief

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # path compression: point at grandparent
        x = parent[x]
    return x

def union(a, b):
    parent[find(a)] = find(b)          # one chief bows to the other

for a, b in [(0, 1), (1, 2), (3, 4), (5, 6), (6, 7), (8, 9)]:
    union(a, b)
    print(f"union({a},{b})")

groups = {}
for x in range(10):
    groups.setdefault(find(x), []).append(x)
print("components      :", sorted(groups.values()))
print("connected(0, 2) :", find(0) == find(2))
print("connected(0, 3) :", find(0) == find(3))

union(0,1)
union(1,2)
union(3,4)
union(5,6)
union(6,7)
union(8,9)
components      : [[0, 1, 2], [3, 4], [5, 6, 7], [8, 9]]
connected(0, 2) : True
connected(0, 3) : False


## 195 · Array vs list vs map

Default to a dynamic array; take a hashmap when keyed lookup dominates; take a linked structure only when O(1) mid-sequence splicing truly matters.

*Three containers on the decision counter: a numbered pill organizer (array — position is meaning), a valet board of name-tagged hooks (map — find by name), and a paper chain you can snip anywhere (list — splice).*

**Watch:** the same membership question against the same 20,000 values — the list scans, the set hashes. The exact times vary per machine, but the ratio is brutal and grows with n. When lookup dominates, take the map.

In [10]:
import timeit

n = 20_000
data_list = list(range(n))
data_set = set(data_list)
needle = n - 1  # worst case for the list: the last element

t_list = timeit.timeit(lambda: needle in data_list, number=2_000)
t_set = timeit.timeit(lambda: needle in data_set, number=2_000)

print(f"x in list : {t_list:.4f}s  (each probe scans up to {n:,} items)")
print(f"x in set  : {t_set:.4f}s  (each probe is one hash lookup)")
print(f"set wins by ~{t_list / t_set:,.0f}x here, and the gap widens as n grows.")

x in list : 0.2204s  (each probe scans up to 20,000 items)
x in set  : 0.0001s  (each probe is one hash lookup)
set wins by ~2,360x here, and the gap widens as n grows.
